# Macmillan → tabla `avance_macmillan`

Mismo camino que el de Oxford, pero estos archivos son más simples: **una fila por estudiante**
(cuatro promedios), no un desglose por unidad.

| Celda | Qué comprueba |
|---|---|
| 1-4 | Kernel, pandas y lectura de los archivos |
| 5-7 | Las funciones, probadas primero en un archivo |
| 8-10 | Todos los archivos, revisión y dataset final |
| 11-15 | Tabla en SQL Server y carga que **actualiza** en vez de duplicar |

In [1]:
# 1. ¿El kernel responde? Si esta celda no termina, el problema es el kernel, no el codigo.
import time
from pathlib import Path

# Marca de version del cuaderno. Si lo que ves aqui NO coincide con la ultima correccion,
# estas mirando una copia vieja: cierra la pestaña sin guardar y vuelve a abrir el archivo.
CUADERNO = "corregido el 2026-09-22 15:03"

ORIGEN = Path.cwd() / "Descargas" / "Macmillan"      # la carpeta entera (o un archivo suelto)
GUARDAR_EN = Path.cwd() / "avance_macmillan.csv"     # respaldo de lo que se cargo

print("=" * 60)
print(f"  CUADERNO MACMILLAN  ({CUADERNO})")
print("=" * 60)
print("kernel vivo:", time.strftime("%H:%M:%S"))
print("origen:", ORIGEN)
print("existe:", ORIGEN.exists(), "| es carpeta:", ORIGEN.is_dir())

  CUADERNO MACMILLAN  (corregido el 2026-09-22 15:03)
kernel vivo: 17:00:50
origen: C:\Users\sebastian.mendez\OneDrive - CENTRO COLOMBO AMERICANO\Escritorio\Automatizacion Oxford\Descargas\Macmillan
existe: True | es carpeta: True


In [2]:
# 2. Cargar pandas (la primera vez puede tardar unos segundos)
inicio = time.time()
import re

import pandas as pd

print(f"pandas {pd.__version__} listo en {time.time() - inicio:.1f} s")

pandas 3.0.6 listo en 0.4 s


In [3]:
# 3. Que archivos vamos a leer
EXTENSIONES = (".csv", ".xlsx", ".xls", ".xlsm")   # lo demas que haya en la carpeta se ignora

archivos = ([a for a in sorted(ORIGEN.iterdir()) if a.suffix.lower() in EXTENSIONES]
            if ORIGEN.is_dir() else [ORIGEN])
print(f"{len(archivos)} archivo(s)")
for archivo in archivos[:12]:
    print(f"  {archivo.name}")
if len(archivos) > 12:
    print(f"  ... y {len(archivos) - 12} mas")

47 archivo(s)
  2026-09-22_1544_Speak_Your_Mind_Level_1_A1.1-8AM02_JAIME_GARZÓN.csv
  2026-09-22_1544_Speak_Your_Mind_Level_1_A1.2-1PM01_SAN_CAYETANO.csv
  2026-09-22_1544_Speak_Your_Mind_Level_1_A1.2-1PM06_EL_UVAL.csv
  2026-09-22_1544_Speak_Your_Mind_Level_1_TCP_A1_1-130PM_TC_LMJ_05.csv
  2026-09-22_1548_Speak_Your_Mind_Level_1_A1.1-12PM04_JOSÉ_CELESTINO_MUTIS.csv
  2026-09-22_1548_Speak_Your_Mind_Level_1_A1.1-12PM05_JOSÉ_CELESTINO_MUTIS.csv
  2026-09-22_1549_Speak_Your_Mind_Level_1_A1.2-8AM03_EL_UVAL.csv
  2026-09-22_1549_Speak_Your_Mind_Level_1_TCP_A1_1-130PM_TC_LMJ_03.csv
  2026-09-22_1549_Speak_Your_Mind_Level_1_TCP_A1_1-130PM_TC_LMJ_08.csv
  2026-09-22_1601_Speak_Your_Mind_Level_1_A1.1-8AM02_JAIME_GARZÓN.csv
  2026-09-22_1601_Speak_Your_Mind_Level_1_A1.2-1PM01_SAN_CAYETANO.csv
  2026-09-22_1601_Speak_Your_Mind_Level_1_A1.2-1PM06_EL_UVAL.csv
  ... y 35 mas


In [4]:
# 4. Leer cada archivo (encabezado normal, una fila por estudiante)
def codificacion_de(ruta):
    """Adivina la codificacion por la firma (BOM). Macmillan exporta en UTF-8 sin firma."""
    firma = ruta.read_bytes()[:3]
    if firma.startswith((b"\xff\xfe", b"\xfe\xff")):
        return "utf-16"
    if firma.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig"
    return "utf-8"


def leer(ruta):
    """Devuelve el archivo con sus columnas, todo como texto (se convierte despues)."""
    if ruta.suffix.lower() in (".xlsx", ".xls", ".xlsm"):
        return pd.read_excel(ruta, dtype=object)
    return pd.read_csv(ruta, dtype=object, encoding=codificacion_de(ruta),
                       sep=None, engine="python")


crudos = {}
for archivo in archivos:
    inicio = time.time()
    crudo = leer(archivo)
    crudos[archivo.name] = crudo
    print(f"  {archivo.name[:60]:<62} {crudo.shape[0]:>3} filas x {crudo.shape[1]} col"
          f"  ({time.time() - inicio:.1f} s)")

print("\ncolumnas del primero:")
print(list(crudos[archivos[0].name].columns))

  2026-09-22_1544_Speak_Your_Mind_Level_1_A1.1-8AM02_JAIME_GAR    25 filas x 9 col  (0.0 s)
  2026-09-22_1544_Speak_Your_Mind_Level_1_A1.2-1PM01_SAN_CAYET    26 filas x 9 col  (0.0 s)
  2026-09-22_1544_Speak_Your_Mind_Level_1_A1.2-1PM06_EL_UVAL.c    26 filas x 9 col  (0.0 s)
  2026-09-22_1544_Speak_Your_Mind_Level_1_TCP_A1_1-130PM_TC_LM     3 filas x 9 col  (0.0 s)
  2026-09-22_1548_Speak_Your_Mind_Level_1_A1.1-12PM04_JOSÉ_CEL    26 filas x 9 col  (0.0 s)
  2026-09-22_1548_Speak_Your_Mind_Level_1_A1.1-12PM05_JOSÉ_CEL    26 filas x 9 col  (0.0 s)
  2026-09-22_1549_Speak_Your_Mind_Level_1_A1.2-8AM03_EL_UVAL.c    26 filas x 9 col  (0.0 s)
  2026-09-22_1549_Speak_Your_Mind_Level_1_TCP_A1_1-130PM_TC_LM     8 filas x 9 col  (0.0 s)
  2026-09-22_1549_Speak_Your_Mind_Level_1_TCP_A1_1-130PM_TC_LM     6 filas x 9 col  (0.0 s)
  2026-09-22_1601_Speak_Your_Mind_Level_1_A1.1-8AM02_JAIME_GAR    25 filas x 9 col  (0.0 s)
  2026-09-22_1601_Speak_Your_Mind_Level_1_A1.2-1PM01_SAN_CAYET    26 filas x 9 c

In [5]:
# 5. Las funciones de apoyo
COLUMNAS = {                       # como se llama en Macmillan -> como se llama en la tabla
    "Course Name": "course_name",
    "Class Name": "class_name",
    "Student Name": "student_name",
    "User Name": "user_name",
    "Attempt": "attempt",
    "Overall Course Average": "overall_course_average",
    "Coursework Average": "coursework_average",
    "Homework Average": "homework_average",
    "Assessment Average": "assessment_average",
}
PROMEDIOS = ["overall_course_average", "coursework_average",
             "homework_average", "assessment_average"]
TEXTOS = ["course_name", "class_name", "student_name", "user_name", "attempt"]

#   A1.1-12PM04 JOSÉ CELESTINO MUTIS          TCP_A1_1-130PM_TC_LMJ_05
#   \__/ \__/\/ \__________________/               \__/ \___/ \____/\/
#  clase grupo cod      colegio                   clase grupo colegio cod
PATRON_CLASE = re.compile(r"""(?ix)
    (?P<clase>[a-z]{1,3}\s*\d+(?:[.\-_]\d+)?)     # A1.1, A1_1, B1-2
    [\s\-_]*
    (?P<grupo>\d{1,4}\s*[ap]\.?m\.?)              # 12PM, 8AM, 130PM (1:30 p.m.)
    (?P<resto>.*)$
""")


def texto(serie):
    """Serie de texto limpio y sin NaN."""
    return serie.astype("string").fillna("").str.strip()


def a_porcentaje(serie):
    """'53%' -> 53.0   |   'No activities completed', 'No homework graded' -> NaN"""
    return pd.to_numeric(texto(serie).str.replace("%", "", regex=False),
                         errors="coerce").round(2)


def partes_de_clase(class_name):
    """Saca clase, grupo, codigo y colegio de 'A1.1-12PM04 JOSE CELESTINO MUTIS'."""
    limpio = " ".join(str(class_name).split())
    hallado = PATRON_CLASE.search(limpio)        # search: aguanta prefijos como 'TCP_'
    if not hallado:
        return {"clase": None, "codigo": None, "grupo": None, "colegio": None}

    resto = hallado["resto"].strip(" -_")
    pegado = re.match(r"^(\d+)", resto)          # lo normal: el codigo va pegado al grupo
    if pegado:
        codigo, colegio = pegado[1], resto[pegado.end():]
    else:                                        # variantes con el codigo al final
        final = re.search(r"(\d+)\s*$", resto)
        codigo = final[1] if final else None
        colegio = resto[:final.start()] if final else resto
    return {
        "clase": re.sub(r"[\s_\-]", ".", hallado["clase"].strip()).upper(),   # A1-1 -> A1.1
        "codigo": codigo,                        # texto: un 04 no se vuelve 4
        "grupo": re.sub(r"[\s.]", "", hallado["grupo"]).upper(),
        "colegio": " ".join(re.sub(r"[_\-]+", " ", colegio).split()).upper() or None,
    }


print("porcentajes ->", a_porcentaje(pd.Series(
    ["53%", "0%", "No activities completed", "No homework graded", "No assessment graded"])).tolist())
for ejemplo in ("A1.1-12PM04 JOSÉ CELESTINO MUTIS", "A1.1-1PM02 SAN CAYETANO",
                "TCP_A1_1-130PM_TC_LMJ_05", "B1.2-5PM28", "Clase rara sin codigo"):
    print(f"  {ejemplo:<36} -> {partes_de_clase(ejemplo)}")

porcentajes -> [53, 0, <NA>, <NA>, <NA>]
  A1.1-12PM04 JOSÉ CELESTINO MUTIS     -> {'clase': 'A1.1', 'codigo': '04', 'grupo': '12PM', 'colegio': 'JOSÉ CELESTINO MUTIS'}
  A1.1-1PM02 SAN CAYETANO              -> {'clase': 'A1.1', 'codigo': '02', 'grupo': '1PM', 'colegio': 'SAN CAYETANO'}
  TCP_A1_1-130PM_TC_LMJ_05             -> {'clase': 'A1.1', 'codigo': '05', 'grupo': '130PM', 'colegio': 'TC LMJ'}
  B1.2-5PM28                           -> {'clase': 'B1.2', 'codigo': '28', 'grupo': '5PM', 'colegio': None}
  Clase rara sin codigo                -> {'clase': None, 'codigo': None, 'grupo': None, 'colegio': None}


In [6]:
# 6. Un archivo de ejemplo, para ver la transformacion antes de aplicarla a todos
def a_tabla(crudo, origen=""):
    """Renombra las columnas, convierte los promedios y saca clase/grupo/codigo/colegio."""
    faltan = [c for c in COLUMNAS if c not in crudo.columns]
    if faltan:
        raise ValueError(f"A '{origen}' le faltan columnas: {faltan}")

    tabla = crudo[list(COLUMNAS)].rename(columns=COLUMNAS).copy()
    for columna in TEXTOS:
        tabla[columna] = texto(tabla[columna])
    for columna in PROMEDIOS:
        tabla[columna] = a_porcentaje(tabla[columna])

    partes = pd.DataFrame([partes_de_clase(c) for c in tabla["class_name"]], index=tabla.index)
    tabla = pd.concat([tabla, partes], axis=1)
    tabla = tabla[tabla["student_name"].ne("")]      # filas vacias al final del archivo
    tabla["origen"] = origen
    return tabla.reset_index(drop=True)


ejemplo = a_tabla(crudos[archivos[0].name], archivos[0].name)
print(f"{len(ejemplo)} filas")
ejemplo.head(10)

25 filas


,course_name,class_name,student_name,user_name,attempt,overall_course_average,coursework_average,homework_average,assessment_average,clase,codigo,grupo,colegio,origen
0,Speak Your Mind Level 1,A1.1-8AM02 JAIME GARZÓN,Alex Esteban Beltrán Martines,abeltránmartines079,First,0,<NA>,0,<NA>,A1.1,02,8AM,JAIME GARZÓN,2026-09-22_1544_Speak_Your_Mind_Level_1_A1.1-8...
1,Speak Your Mind Level 1,A1.1-8AM02 JAIME GARZÓN,Andrea Dayana Chingate Gamba,achingategamba047,First,72,72,72,<NA>,A1.1,02,8AM,JAIME GARZÓN,2026-09-22_1544_Speak_Your_Mind_Level_1_A1.1-8...
2,Speak Your Mind Level 1,A1.1-8AM02 JAIME GARZÓN,Anyi Mariana Silva Medina,asilvamedina389,First,34,33,36,<NA>,A1.1,02,8AM,JAIME GARZÓN,2026-09-22_1544_Speak_Your_Mind_Level_1_A1.1-8...
3,Speak Your Mind Level 1,A1.1-8AM02 JAIME GARZÓN,Cristian Andrey Coy Ibáñez,ccoyibáñez059,First,76,76,76,<NA>,A1.1,02,8AM,JAIME GARZÓN,2026-09-22_1544_Speak_Your_Mind_Level_1_A1.1-8...
4,Speak Your Mind Level 1,A1.1-8AM02 JAIME GARZÓN,Derick Damian Contreras Romero,dcontrerasromero278,First,63,63,63,<NA>,A1.1,02,8AM,JAIME GARZÓN,2026-09-22_1544_Speak_Your_Mind_Level_1_A1.1-8...
5,Speak Your Mind Level 1,A1.1-8AM02 JAIME GARZÓN,Gabriela Mayorga Quevedo,gmayorgaquevedo347,First,0,<NA>,0,<NA>,A1.1,02,8AM,JAIME GARZÓN,2026-09-22_1544_Speak_Your_Mind_Level_1_A1.1-8...
6,Speak Your Mind Level 1,A1.1-8AM02 JAIME GARZÓN,Heilen Valentina Espitia Ibañez,hespitiaibañez078,First,73,73,73,<NA>,A1.1,02,8AM,JAIME GARZÓN,2026-09-22_1544_Speak_Your_Mind_Level_1_A1.1-8...
7,Speak Your Mind Level 1,A1.1-8AM02 JAIME GARZÓN,Helen Julieth González Roman,hgonzálezroman456,First,82,82,82,<NA>,A1.1,02,8AM,JAIME GARZÓN,2026-09-22_1544_Speak_Your_Mind_Level_1_A1.1-8...
8,Speak Your Mind Level 1,A1.1-8AM02 JAIME GARZÓN,Henry Gabriel Simbaqueva Valbuena,hsimbaquevavalbuena235,First,52,52,<NA>,<NA>,A1.1,02,8AM,JAIME GARZÓN,2026-09-22_1544_Speak_Your_Mind_Level_1_A1.1-8...
9,Speak Your Mind Level 1,A1.1-8AM02 JAIME GARZÓN,Jeimmy Judely Sanabria Gutiérrez,jsanabriagutiérrez235,First,37,74,0,<NA>,A1.1,02,8AM,JAIME GARZÓN,2026-09-22_1544_Speak_Your_Mind_Level_1_A1.1-8...


In [7]:
# 7. Todos los archivos, quedandose con la descarga MAS RECIENTE de cada estudiante.
#    La carpeta guarda el historico: la misma clase puede estar bajada varios dias.
FECHA_EN_NOMBRE = re.compile(r"^(\d{4}-\d{2}-\d{2})_(\d{2})(\d{2})_")
LLAVE_FILA = ["course_name", "class_name", "user_name", "attempt"]


def descargado_en(ruta):
    """La fecha y hora que el bot pone al inicio del nombre; si no esta, la del archivo."""
    hallado = FECHA_EN_NOMBRE.match(ruta.name)
    if hallado:
        return pd.Timestamp(f"{hallado[1]} {hallado[2]}:{hallado[3]}")
    return pd.Timestamp(ruta.stat().st_mtime, unit="s")


partes = []
for archivo in archivos:
    try:
        tabla = a_tabla(crudos[archivo.name], archivo.name)
        tabla["descargado_en"] = descargado_en(archivo)
        partes.append(tabla)
    except Exception as error:
        print(f"  SE SALTA {archivo.name}: {error}")

todo = pd.concat(partes, ignore_index=True)
avance = (todo.sort_values("descargado_en")
              .drop_duplicates(subset=LLAVE_FILA, keep="last")   # se queda la ultima descarga
              .reset_index(drop=True))

print(f"{len(todo):,} filas leidas de {len(partes)} archivo(s)")
print(f"{len(todo) - len(avance):,} eran descargas repetidas de la misma clase")
print(f"{len(avance):,} filas finales (una por estudiante, clase e intento)")
print("\nfilas por dia de descarga:")
print(todo.groupby(todo["descargado_en"].dt.date).size().rename("filas"))
avance.head()

808 filas leidas de 47 archivo(s)
252 eran descargas repetidas de la misma clase
556 filas finales (una por estudiante, clase e intento)

filas por dia de descarga:
descargado_en
2026-09-22    808
Name: filas, dtype: int64


,course_name,class_name,student_name,user_name,attempt,overall_course_average,coursework_average,homework_average,assessment_average,clase,codigo,grupo,colegio,origen,descargado_en
0,Speak Your Mind Level 1,A1.2-1PM06 EL UVAL,Breidy Yisell Palma Riaño,bpalmariaño466,First,<NA>,<NA>,<NA>,<NA>,A1.2,06,1PM,EL UVAL,2026-09-22_1634_Speak_Your_Mind_Level_1_A1.2-1...,2026-09-22 16:34:00
1,Speak Your Mind Level 1,A1.2-1PM06 EL UVAL,Angie Sofia Laguna Cabezas,alagunacabezas779,First,<NA>,<NA>,<NA>,<NA>,A1.2,06,1PM,EL UVAL,2026-09-22_1634_Speak_Your_Mind_Level_1_A1.2-1...,2026-09-22 16:34:00
2,Speak Your Mind Level 1,A1.2-1PM06 EL UVAL,Angel David Ospina Zamora,aospinazamora456,First,<NA>,<NA>,<NA>,<NA>,A1.2,06,1PM,EL UVAL,2026-09-22_1634_Speak_Your_Mind_Level_1_A1.2-1...,2026-09-22 16:34:00
3,Speak Your Mind Level 1,A1.2-1PM06 EL UVAL,Anderson Alexis Caro Solano,acarosolano567,First,<NA>,<NA>,<NA>,<NA>,A1.2,06,1PM,EL UVAL,2026-09-22_1634_Speak_Your_Mind_Level_1_A1.2-1...,2026-09-22 16:34:00
4,Speak Your Mind Level 1,A1.2-1PM06 EL UVAL,Brenda Valentina Palma Riaño,bpalmariaño448,First,<NA>,<NA>,<NA>,<NA>,A1.2,06,1PM,EL UVAL,2026-09-22_1634_Speak_Your_Mind_Level_1_A1.2-1...,2026-09-22 16:34:00


In [8]:
# 8. Revision del dataset
print(avance.dtypes, "\n")
print("estudiantes:", avance["user_name"].nunique(),
      "| clases:", avance["class_name"].nunique(),
      "| cursos:", avance["course_name"].nunique(),
      "| colegios:", avance["colegio"].nunique())
print("\nintentos:\n", avance["attempt"].value_counts())
print("\nvacios por columna:\n", avance.isna().sum())
print("\nrangos:")
for columna in PROMEDIOS:
    print(f"  {columna:<24} min {avance[columna].min()}  max {avance[columna].max()}"
          f"  vacios {avance[columna].isna().sum()}")

sin_clase = avance["clase"].isna().sum()
if sin_clase:
    print(f"\nOJO: {sin_clase} filas sin clase; revisa esos Class Name:")
    display(avance.loc[avance["clase"].isna(), ["class_name", "origen"]].drop_duplicates())

# Un estudiante no deberia estar dos veces en la misma clase con el mismo intento
repetidas = avance.duplicated(subset=["course_name", "class_name", "user_name", "attempt"]).sum()
print(f"\nfilas repetidas (curso + clase + usuario + intento): {repetidas}")

course_name                       string
class_name                        string
student_name                      string
user_name                         string
attempt                           string
overall_course_average           Float64
coursework_average               Float64
homework_average                 Float64
assessment_average               Float64
clase                             object
codigo                            object
grupo                             object
colegio                           object
origen                               str
descargado_en             datetime64[us]
dtype: object 

estudiantes: 542 | clases: 34 | cursos: 1 | colegios: 14

intentos:
 attempt
First    556
Name: count, dtype: Int64

vacios por columna:
 course_name                 0
class_name                  0
student_name                0
user_name                   0
attempt                     0
overall_course_average    199
coursework_average        211
homework_average     

,class_name,origen
545,A1-3 MCE MEDIADO -- PRIME-7PM72-OCTUBRE 2024,2026-09-22_1658_Speak_Your_Mind_Level_1_A1-3_M...



filas repetidas (curso + clase + usuario + intento): 0


In [9]:
# 9. Dataset final, con las columnas de la tabla
ahora = pd.Timestamp.now()
avance_macmillan = avance.copy()
avance_macmillan["created_at"] = ahora      # cuando aparecio (solo cuenta al insertar)
avance_macmillan["visto_en"] = ahora        # cuando se vio por ultima vez en una descarga
avance_macmillan = avance_macmillan[["course_name", "class_name", "student_name", "user_name",
                                     "attempt", "overall_course_average", "coursework_average",
                                     "homework_average", "assessment_average",
                                     "created_at", "visto_en",
                                     "clase", "codigo", "grupo", "colegio"]]

avance_macmillan.to_csv(GUARDAR_EN, index=False, encoding="utf-8-sig")
print(f"{len(avance_macmillan):,} filas guardadas en {GUARDAR_EN}")
print("lote (created_at):", avance_macmillan["created_at"].iloc[0])
avance_macmillan.head(20)

556 filas guardadas en C:\Users\sebastian.mendez\OneDrive - CENTRO COLOMBO AMERICANO\Escritorio\Automatizacion Oxford\avance_macmillan.csv
lote (created_at): 2026-09-22 17:00:51.365588


,course_name,class_name,student_name,user_name,attempt,overall_course_average,coursework_average,homework_average,assessment_average,created_at,visto_en,clase,codigo,grupo,colegio
0,Speak Your Mind Level 1,A1.2-1PM06 EL UVAL,Breidy Yisell Palma Riaño,bpalmariaño466,First,<NA>,<NA>,<NA>,<NA>,2026-09-22 17:00:51.365588,2026-09-22 17:00:51.365588,A1.2,06,1PM,EL UVAL
1,Speak Your Mind Level 1,A1.2-1PM06 EL UVAL,Angie Sofia Laguna Cabezas,alagunacabezas779,First,<NA>,<NA>,<NA>,<NA>,2026-09-22 17:00:51.365588,2026-09-22 17:00:51.365588,A1.2,06,1PM,EL UVAL
2,Speak Your Mind Level 1,A1.2-1PM06 EL UVAL,Angel David Ospina Zamora,aospinazamora456,First,<NA>,<NA>,<NA>,<NA>,2026-09-22 17:00:51.365588,2026-09-22 17:00:51.365588,A1.2,06,1PM,EL UVAL
3,Speak Your Mind Level 1,A1.2-1PM06 EL UVAL,Anderson Alexis Caro Solano,acarosolano567,First,<NA>,<NA>,<NA>,<NA>,2026-09-22 17:00:51.365588,2026-09-22 17:00:51.365588,A1.2,06,1PM,EL UVAL
4,Speak Your Mind Level 1,A1.2-1PM06 EL UVAL,Brenda Valentina Palma Riaño,bpalmariaño448,First,<NA>,<NA>,<NA>,<NA>,2026-09-22 17:00:51.365588,2026-09-22 17:00:51.365588,A1.2,06,1PM,EL UVAL
5,Speak Your Mind Level 1,A1.2-1PM01 SAN CAYETANO,Yubis Vique Sanchez,yviquesanchez559,First,70.0,70.0,<NA>,<NA>,2026-09-22 17:00:51.365588,2026-09-22 17:00:51.365588,A1.2,01,1PM,SAN CAYETANO
6,Speak Your Mind Level 1,A1.2-1PM01 SAN CAYETANO,Yoseph Sebastian Huerfano Herreño,yhuerfanoherreño028,First,77.0,77.0,<NA>,<NA>,2026-09-22 17:00:51.365588,2026-09-22 17:00:51.365588,A1.2,01,1PM,SAN CAYETANO
7,Speak Your Mind Level 1,A1.2-1PM01 SAN CAYETANO,Julian David Bogota Valencia,jbogotavalencia467,First,55.0,55.0,<NA>,<NA>,2026-09-22 17:00:51.365588,2026-09-22 17:00:51.365588,A1.2,01,1PM,SAN CAYETANO
8,Speak Your Mind Level 1,A1.2-1PM01 SAN CAYETANO,Sara Valentina Mora Piamonte,smorapiamonte469,First,92.0,92.0,<NA>,<NA>,2026-09-22 17:00:51.365588,2026-09-22 17:00:51.365588,A1.2,01,1PM,SAN CAYETANO
9,Speak Your Mind Level 1,A1.2-1PM01 SAN CAYETANO,Maria Jose Gonzalez Sanchez,mgonzalezsanchez779,First,<NA>,<NA>,<NA>,<NA>,2026-09-22 17:00:51.365588,2026-09-22 17:00:51.365588,A1.2,01,1PM,SAN CAYETANO


## Carga a SQL Server

La tabla guarda **una fila por estudiante y clase**, y se actualiza en cada descarga.
`id` lo genera la tabla.

In [10]:
# 10. Conexion (autenticacion de Windows, no se escriben usuario ni clave)
import pyodbc
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

SERVIDOR = "CCA-CENT-R5186"          # instancia con nombre: "CCA-CENT-R5186\\SQLEXPRESS"
BASE = "rendimiento_academico"
TABLA = "avance_macmillan"
STAGE = f"{TABLA}_stage"

PREFERIDOS = ["ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server",
              "ODBC Driver 13 for SQL Server", "SQL Server Native Client 11.0", "SQL Server"]
instalados = pyodbc.drivers()
driver = next((d for d in PREFERIDOS if d in instalados), None)
if driver is None:
    raise RuntimeError(f"No hay driver de SQL Server. Instalados: {instalados}")

MODERNO = driver.startswith("ODBC Driver")
cadena = (f"DRIVER={{{driver}}};SERVER={SERVIDOR};DATABASE={BASE};"
          "Trusted_Connection=yes;TrustServerCertificate=yes")
motor = create_engine(URL.create("mssql+pyodbc", query={"odbc_connect": cadena}),
                      fast_executemany=MODERNO)

with motor.connect() as conexion:
    quien = conexion.execute(text("SELECT SUSER_NAME(), DB_NAME()")).one()
    columnas_hoy = pd.read_sql(text("""
        SELECT COLUMN_NAME AS columna, DATA_TYPE AS tipo, CHARACTER_MAXIMUM_LENGTH AS largo
        FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = :tabla ORDER BY ORDINAL_POSITION
    """), conexion, params={"tabla": TABLA})
print(f"driver elegido: {driver}" + ("" if MODERNO else "   <- viejo: la carga ira mas lenta"))
print(f"conectado como {quien[0]} a {quien[1]}")
print(f"\nla tabla dbo.{TABLA} tiene hoy estas columnas:")
display(columnas_hoy)

driver elegido: ODBC Driver 18 for SQL Server
conectado como COLOMBO\sebastian.mendez a rendimiento_academico

la tabla dbo.avance_macmillan tiene hoy estas columnas:


,columna,tipo,largo
0,id,int,NaN
1,course_name,varchar,100.0
2,class_name,varchar,100.0
3,student_name,varchar,150.0
4,user_name,varchar,100.0
5,attempt,varchar,30.0
6,overall_course_average,float,NaN
7,coursework_average,float,NaN
8,homework_average,float,NaN
9,assessment_average,float,NaN


In [11]:
# 11. Dejar la tabla lista: agrega lo que falte y prepara la actualizacion.
#     Se puede correr las veces que quieras: si ya esta hecho, no hace nada.

# La llave natural: un estudiante, en una clase, con un intento
LLAVE = """CONCAT(course_name, '|', class_name, '|', user_name, '|', attempt)"""

COLUMNAS_SQL = ["course_name", "class_name", "student_name", "user_name", "attempt",
                "overall_course_average", "coursework_average", "homework_average",
                "assessment_average", "created_at", "visto_en",
                "clase", "codigo", "grupo", "colegio"]
LISTA_SQL = ", ".join(COLUMNAS_SQL)

CREAR = f"""CREATE TABLE dbo.{TABLA} (
    id INT IDENTITY(1,1) PRIMARY KEY,
    course_name VARCHAR(100) NULL, class_name VARCHAR(100) NULL,
    student_name VARCHAR(150) NULL, user_name VARCHAR(100) NULL, attempt VARCHAR(30) NULL,
    overall_course_average FLOAT NULL, coursework_average FLOAT NULL,
    homework_average FLOAT NULL, assessment_average FLOAT NULL)"""

FALTANTES = {                      # lo que la tabla original no traia
    "created_at": "DATETIME2 NULL",     # cuando aparecio por primera vez
    "updated_at": "DATETIME2 NULL",     # cuando cambio de valor por ultima vez
    "visto_en": "DATETIME2 NULL",       # cuando se vio por ultima vez en una descarga
    "clase": "VARCHAR(30) NULL",
    "codigo": "VARCHAR(30) NULL",
    "grupo": "VARCHAR(30) NULL",
    "colegio": "VARCHAR(100) NULL",
}

with motor.begin() as conexion:
    conexion.exec_driver_sql(f"IF OBJECT_ID('dbo.{TABLA}') IS NULL EXEC('{CREAR}');")
    for columna, tipo in FALTANTES.items():
        conexion.exec_driver_sql(f"IF COL_LENGTH('dbo.{TABLA}', '{columna}') IS NULL "
                                 f"ALTER TABLE dbo.{TABLA} ADD {columna} {tipo};")

    # Los promedios deben ser numero: como texto no se pueden promediar en Power BI
    filas = conexion.exec_driver_sql(f"SELECT COUNT(*) FROM dbo.{TABLA}").scalar()
    de_texto = [c for c, t in conexion.exec_driver_sql(
        "SELECT COLUMN_NAME, DATA_TYPE FROM INFORMATION_SCHEMA.COLUMNS "
        f"WHERE TABLE_NAME = '{TABLA}'").all()
        if c in PROMEDIOS and t.lower() in ("varchar", "nvarchar", "char", "nchar")]
    if de_texto and filas == 0:
        for columna in de_texto:
            conexion.exec_driver_sql(f"ALTER TABLE dbo.{TABLA} ALTER COLUMN {columna} FLOAT NULL")
        print("promedios pasados de texto a numero:", de_texto)
    elif de_texto:
        print(f"OJO: {de_texto} son de texto y la tabla ya tiene {filas:,} filas.")
        print("Para poder promediarlos en Power BI, vacia la tabla y vuelve a correr esta celda:")
        print(f"    DELETE FROM dbo.{TABLA};")

    # Huella de la llave: 32 bytes en vez de comparar 4 columnas, y sin problemas de NULL
    conexion.exec_driver_sql(
        f"""IF COL_LENGTH('dbo.{TABLA}', 'clave_hash') IS NULL
                ALTER TABLE dbo.{TABLA} ADD clave_hash
                    AS CONVERT(BINARY(32), HASHBYTES('SHA2_256', {LLAVE})) PERSISTED;""")
    # La tabla de paso se crea con los MISMOS tipos, si no el hash no coincidiria
    conexion.exec_driver_sql(
        f"""IF OBJECT_ID('dbo.{STAGE}') IS NULL
                EXEC('SELECT TOP 0 {LISTA_SQL} INTO dbo.{STAGE} FROM dbo.{TABLA}');""")
    conexion.exec_driver_sql(
        f"""IF COL_LENGTH('dbo.{STAGE}', 'clave_hash') IS NULL
                ALTER TABLE dbo.{STAGE} ADD clave_hash
                    AS CONVERT(BINARY(32), HASHBYTES('SHA2_256', {LLAVE})) PERSISTED;""")

    repetidas_en_tabla = conexion.exec_driver_sql(
        f"SELECT COUNT(*) FROM (SELECT clave_hash FROM dbo.{TABLA} "
        "GROUP BY clave_hash HAVING COUNT(*) > 1) AS x").scalar()
    hay_indice = conexion.exec_driver_sql(
        "SELECT COUNT(*) FROM sys.indexes WHERE name = 'UX_avance_macmillan_clave' "
        f"AND object_id = OBJECT_ID('dbo.{TABLA}')").scalar()
    if hay_indice:
        print("El indice unico ya existe: no se puede duplicar una fila.")
    elif repetidas_en_tabla:
        print(f"OJO: hay {repetidas_en_tabla:,} llaves repetidas de cargas anteriores.")
        print(f"Limpia la tabla y vuelve a correr esta celda:   DELETE FROM dbo.{TABLA};")
    else:
        conexion.exec_driver_sql(
            f"CREATE UNIQUE INDEX UX_avance_macmillan_clave ON dbo.{TABLA} (clave_hash)")
        print("Indice unico creado: desde ahora es imposible duplicar una fila.")

with motor.connect() as conexion:
    antes = conexion.execute(text(f"SELECT COUNT(*) FROM dbo.{TABLA}")).scalar()
print(f"\ntabla dbo.{TABLA}: {antes:,} filas   |   tabla de paso: dbo.{STAGE}")

El indice unico ya existe: no se puede duplicar una fila.

tabla dbo.avance_macmillan: 426 filas   |   tabla de paso: dbo.avance_macmillan_stage


In [12]:
# 12. Dejar el dataset listo para la tabla (NaN -> NULL)
from sqlalchemy.types import DateTime, Float, NVARCHAR

TIPOS = {
    "course_name": NVARCHAR(100), "class_name": NVARCHAR(100), "student_name": NVARCHAR(150),
    "user_name": NVARCHAR(100), "attempt": NVARCHAR(30),
    "overall_course_average": Float(), "coursework_average": Float(),
    "homework_average": Float(), "assessment_average": Float(),
    "created_at": DateTime(), "visto_en": DateTime(),
    "clase": NVARCHAR(30), "codigo": NVARCHAR(30),
    "grupo": NVARCHAR(30), "colegio": NVARCHAR(100),
}

para_sql = avance_macmillan.astype(object).where(avance_macmillan.notna(), None)

faltan = [c for c in TIPOS if c not in para_sql.columns]
sobran = [c for c in para_sql.columns if c not in TIPOS]
print("columnas que faltan:", faltan or "ninguna", "| columnas de mas:", sobran or "ninguna")
print(f"{len(para_sql):,} filas listas para enviar")
print("clases que se van a cargar:")
display(para_sql.groupby(["clase", "grupo", "codigo", "colegio"], dropna=False).size()
        .rename("estudiantes").reset_index())

columnas que faltan: ninguna | columnas de mas: ninguna
556 filas listas para enviar
clases que se van a cargar:


,clase,grupo,codigo,colegio,estudiantes
0,A1.1,0810AM,2026,NZ02 01 NIZA FEBRERO,15
1,A1.1,12PM,04,JOSÉ CELESTINO MUTIS,26
2,A1.1,12PM,05,JOSÉ CELESTINO MUTIS,26
3,A1.1,130PM,01,TC LMJ,5
4,A1.1,130PM,02,TC LMJ,14
5,A1.1,130PM,03,TC LMJ,8
6,A1.1,130PM,04,TC LMJ,4
7,A1.1,130PM,05,TC LMJ,3
8,A1.1,130PM,08,TC LMJ,6
9,A1.1,130PM,09,TC LMJ,4


In [13]:
# 13. La carga: actualiza lo que ya existe e inserta lo nuevo (esta celda escribe en la base)

# 'visto_en' se toca SIEMPRE que la fila venga en la descarga, cambie o no su valor.
# Asi se distingue un estudiante que no avanzo de uno que ya no aparece en la clase.
SQL_VISTO = f"""
UPDATE destino
   SET destino.visto_en = SYSDATETIME()
  FROM dbo.{TABLA} AS destino
  JOIN dbo.{STAGE} AS origen ON origen.clave_hash = destino.clave_hash;
"""

SQL_ACTUALIZAR = f"""
UPDATE destino
   SET destino.student_name           = origen.student_name,
       destino.overall_course_average = origen.overall_course_average,
       destino.coursework_average     = origen.coursework_average,
       destino.homework_average       = origen.homework_average,
       destino.assessment_average     = origen.assessment_average,
       destino.clase                  = origen.clase,
       destino.codigo                 = origen.codigo,
       destino.grupo                  = origen.grupo,
       destino.colegio                = origen.colegio,
       destino.updated_at             = SYSDATETIME()
  FROM dbo.{TABLA} AS destino
  JOIN dbo.{STAGE} AS origen ON origen.clave_hash = destino.clave_hash
 WHERE EXISTS (SELECT origen.overall_course_average, origen.coursework_average,
                      origen.homework_average, origen.assessment_average, origen.student_name
               EXCEPT
               SELECT destino.overall_course_average, destino.coursework_average,
                      destino.homework_average, destino.assessment_average, destino.student_name);
"""

SQL_INSERTAR = f"""
INSERT INTO dbo.{TABLA} ({LISTA_SQL})
SELECT {", ".join("origen." + c for c in COLUMNAS_SQL)}
  FROM dbo.{STAGE} AS origen
 WHERE NOT EXISTS (SELECT 1 FROM dbo.{TABLA} AS d WHERE d.clave_hash = origen.clave_hash);
"""

inicio = time.time()
with motor.begin() as conexion:
    conexion.exec_driver_sql(f"TRUNCATE TABLE dbo.{STAGE}")
    para_sql.to_sql(STAGE, conexion, schema="dbo", if_exists="append", index=False,
                    chunksize=1000 if MODERNO else 200, dtype=TIPOS)

with motor.begin() as conexion:
    repetidas_hoy = conexion.exec_driver_sql(
        f"SELECT COUNT(*) FROM (SELECT clave_hash FROM dbo.{STAGE} "
        "GROUP BY clave_hash HAVING COUNT(*) > 1) AS x").scalar()
    if repetidas_hoy:
        raise ValueError(f"La descarga de hoy trae {repetidas_hoy:,} llaves repetidas. "
                         "No se aplico nada.")
    vistas = conexion.exec_driver_sql(SQL_VISTO).rowcount           # todas las que ya existian
    actualizadas = conexion.exec_driver_sql(SQL_ACTUALIZAR).rowcount
    insertadas = conexion.exec_driver_sql(SQL_INSERTAR).rowcount

print(f"{len(para_sql):,} filas procesadas en {time.time() - inicio:.1f} s")
print(f"  ya estaban (visto_en al dia):   {vistas:,}")
print(f"     de esas, cambiaron de valor: {actualizadas:,}")
print(f"  insertadas (nuevas):            {insertadas:,}")
print(f"  sin cambios:                    {vistas - actualizadas:,}")

556 filas procesadas en 0.1 s
  ya estaban (visto_en al dia):   400
     de esas, cambiaron de valor: 0
  insertadas (nuevas):            156
  sin cambios:                    400


In [14]:
# 14. Comprobar contra la base
with motor.connect() as conexion:
    despues = conexion.execute(text(f"SELECT COUNT(*) FROM dbo.{TABLA}")).scalar()
print(f"antes: {antes:,}  ->  despues: {despues:,}   (filas nuevas: {despues - antes:,})")

promedios = pd.read_sql(text(f"""
    SELECT colegio, clase, grupo, codigo,
           COUNT(*) AS estudiantes,
           ROUND(AVG(overall_course_average), 1) AS promedio_general,
           ROUND(AVG(coursework_average), 1)     AS promedio_curso,
           ROUND(AVG(homework_average), 1)       AS promedio_tareas
    FROM dbo.{TABLA}
    GROUP BY colegio, clase, grupo, codigo
    ORDER BY colegio, clase, grupo, codigo
"""), motor)
display(promedios)

cambios = pd.read_sql(text(f"""
    SELECT CAST(updated_at AS DATE) AS dia, COUNT(*) AS filas_cambiadas
    FROM dbo.{TABLA} WHERE updated_at IS NOT NULL
    GROUP BY CAST(updated_at AS DATE) ORDER BY dia DESC
"""), motor)
print("Cambios por dia (updated_at):")
display(cambios)

# Filas que ya no vienen en la descarga: se vieron por ultima vez antes de esta carga
faltantes = pd.read_sql(text(f"""
    WITH ultima AS (SELECT MAX(visto_en) AS cuando FROM dbo.{TABLA})
    SELECT colegio, clase, grupo, codigo, student_name, user_name,
           CAST(visto_en AS DATE) AS visto_por_ultima_vez
    FROM dbo.{TABLA}, ultima
    WHERE visto_en < DATEADD(MINUTE, -5, ultima.cuando)
    ORDER BY visto_en DESC
"""), motor)
print(f"Filas que dejaron de aparecer en la descarga: {len(faltantes)}")
display(faltantes.head(20))

pd.read_sql(text(f"SELECT TOP 10 * FROM dbo.{TABLA} ORDER BY id DESC"), motor)

antes: 426  ->  despues: 582   (filas nuevas: 156)


,colegio,clase,grupo,codigo,estudiantes,promedio_general,promedio_curso,promedio_tareas
0,NaN,NaN,NaN,NaN,11,69.9,68.4,72.1
1,ÁREA ANDINA,A1.1,5PM,01,9,79.9,80.8,71.0
2,DICIEMBRE 2024,A1.5,0710PM,02,15,73.3,66.6,84.5
3,EL DESTINO,A1.2,11AM,01,38,64.7,63.8,67.2
4,EL UVAL,A1.2,1PM,06,26,77.0,77.0,NaN
5,EL UVAL,A1.2,8AM,03,26,56.7,55.9,57.6
6,EL VERJON,A1.1,1PM,05,28,37.5,53.0,20.1
7,EL VERJÓN,A1.2,1PM,04,24,61.4,61.4,NaN
8,IED TÉCNICO COMERCIAL,TA1.1,8AM,01,4,78.5,78.5,NaN
9,JAIME GARZÓN,A1.1,8AM,02,25,48.8,66.7,41.2


Cambios por dia (updated_at):


,dia,filas_cambiadas


Filas que dejaron de aparecer en la descarga: 26


,colegio,clase,grupo,codigo,student_name,user_name,visto_por_ultima_vez
0,JOSÉ CELESTINO MUTIS,A1.1,12PM,04,Nicxy Mariana Lopez Cifuentes,nlopezcifuentes127,2026-09-22
1,JOSÉ CELESTINO MUTIS,A1.1,12PM,04,Sara Sofia Barrera García,sbarreragarcía026,2026-09-22
2,JOSÉ CELESTINO MUTIS,A1.1,12PM,04,Sharon Eliana Garzón Marín,sgarzónmarín289,2026-09-22
3,JOSÉ CELESTINO MUTIS,A1.1,12PM,04,Shayla Jermany Alcantara Moron,salcantaramoron058,2026-09-22
4,JOSÉ CELESTINO MUTIS,A1.1,12PM,04,Stephany Carolina Manjarres Espitia,smanjarresespitia019,2026-09-22
5,JOSÉ CELESTINO MUTIS,A1.1,12PM,04,YENER SMITH PRIETO RAMIREZ,yprietoramirez122,2026-09-22
6,JOSÉ CELESTINO MUTIS,A1.1,12PM,04,Yuri Tatiana Garzon Tabares,ygarzontabares059,2026-09-22
7,JOSÉ CELESTINO MUTIS,A1.1,12PM,04,Zahira Nicol Moreno Gomez,zmorenogomez126,2026-09-22
8,JOSÉ CELESTINO MUTIS,A1.1,12PM,04,Zorangie Gallo Gutiérrez,zgallogutiérrez017,2026-09-22
9,JOSÉ CELESTINO MUTIS,A1.1,12PM,04,Michell Sofia Garcés Torres,mgarcéstorres023,2026-09-22


,id,course_name,class_name,student_name,user_name,attempt,overall_course_average,coursework_average,homework_average,assessment_average,created_at,updated_at,visto_en,clase,codigo,grupo,colegio,clave_hash
0,582,Speak Your Mind Level 1,A1-3 MCE MEDIADO -- PRIME-7PM72-OCTUBRE 2024,Tatiana Ortega,tatianaortegaserpa,First,73.0,63.0,83.0,None,2026-09-22 17:00:51.365588,None,2026-09-22 17:00:51.365588,None,None,None,None,b'\x85\xc2\xcf\xd4W\x0f\x9b\xe3\x0e\xf1\xcb\xa...
1,581,Speak Your Mind Level 1,A1-3 MCE MEDIADO -- PRIME-7PM72-OCTUBRE 2024,Diego Andres Pinzon Castro,dpinzoncastro058,First,79.0,77.0,82.0,None,2026-09-22 17:00:51.365588,None,2026-09-22 17:00:51.365588,None,None,None,None,b'\xb8\x82\xcf\xda;\xaa2>M\xef\x9am\x97J\xb5z\...
2,580,Speak Your Mind Level 1,A1-3 MCE MEDIADO -- PRIME-7PM72-OCTUBRE 2024,Sofía Guerrero,sguerrero457,First,30.0,60.0,0.0,None,2026-09-22 17:00:51.365588,None,2026-09-22 17:00:51.365588,None,None,None,None,b'\xf8M\xc2\x9f\xcdd\xcc73\xb2s+\x9c\x99\xa6\x...
3,579,Speak Your Mind Level 1,A1-3 MCE MEDIADO -- PRIME-7PM72-OCTUBRE 2024,Alejandra Navas,AlejaNavas,First,66.0,63.0,68.0,None,2026-09-22 17:00:51.365588,None,2026-09-22 17:00:51.365588,None,None,None,None,b'I\xc6p0p\x88\x1aN\xef`\xbc3\xab%[E\xb5\xb4\x...
4,578,Speak Your Mind Level 1,A1-3 MCE MEDIADO -- PRIME-7PM72-OCTUBRE 2024,Alejo_Brx,a778039,First,72.0,80.0,63.0,None,2026-09-22 17:00:51.365588,None,2026-09-22 17:00:51.365588,None,None,None,None,b'\x99O\xed\xc0\x8a\x93P!\xa8\xc1lg\xb7\xdbwUW...
5,577,Speak Your Mind Level 1,A1-3 MCE MEDIADO -- PRIME-7PM72-OCTUBRE 2024,Camilo Andrés Millán,cmillán488,First,68.0,68.0,68.0,None,2026-09-22 17:00:51.365588,None,2026-09-22 17:00:51.365588,None,None,None,None,b'G\x08\xd2\x1b%U\x1a\xa3\xfd\xe2[\x97(C\x99\x...
6,576,Speak Your Mind Level 1,A1-3 MCE MEDIADO -- PRIME-7PM72-OCTUBRE 2024,Andres Turga,AndresT99,First,88.0,82.0,94.0,None,2026-09-22 17:00:51.365588,None,2026-09-22 17:00:51.365588,None,None,None,None,b'\xe1^\x7f\xa3\xb8\xe8r\xdbJ\x0e\xf5O\x10\xe9...
7,575,Speak Your Mind Level 1,A1-3 MCE MEDIADO -- PRIME-7PM72-OCTUBRE 2024,Jhordan Romero Daza,Jhor1691,First,75.0,65.0,84.0,None,2026-09-22 17:00:51.365588,None,2026-09-22 17:00:51.365588,None,None,None,None,b'\xe2\xc6C\x9de\xac\x84Q\xc3l\x0f\xedP\xa7\xd...
8,574,Speak Your Mind Level 1,A1-3 MCE MEDIADO -- PRIME-7PM72-OCTUBRE 2024,Maria Angelica Rodriguez Castro,Angelica2017,First,77.0,72.0,82.0,None,2026-09-22 17:00:51.365588,None,2026-09-22 17:00:51.365588,None,None,None,None,"b""_V\\\xd0'\xb6\xde\x1b\xb0\x01fL\x01T\xe0hd!:..."
9,573,Speak Your Mind Level 1,A1-3 MCE MEDIADO -- PRIME-7PM72-OCTUBRE 2024,Paola Andrea Arias Colorado,1006508155,First,78.0,59.0,97.0,None,2026-09-22 17:00:51.365588,None,2026-09-22 17:00:51.365588,None,None,None,None,b'/\xf0\xf7\xc5\xc2\x85\xed\xb3I\x9e(W\xba\x85...
